In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

restaurants = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/processed/restaurants_clean.csv")

taxonomy = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/reference/category_mapping.csv", encoding="cp1252")

In [2]:
selected_restaurant = restaurants[
    (restaurants["name"] == "Sonic Drive-In") &
    (restaurants["city"] == "Ashland City") &
    (restaurants["state"] == "TN")
].iloc[0]

In [3]:
selected_state = selected_restaurant["state"]

print(selected_state)

TN


In [4]:
subset_restaurants = restaurants[
    restaurants["state"] == selected_state
].copy()

subset_restaurants.reset_index(drop=True, inplace=True)


In [5]:
restaurants = subset_restaurants.copy()

In [6]:
taxonomy_keep = taxonomy[taxonomy["Keep"] == "Yes"]

In [7]:
mapping = taxonomy_keep.set_index("Category").to_dict("index")

In [8]:
categories = [
    c.strip()
    for c in restaurants.iloc[0]["categories"].split(",")
]

In [9]:
for c in categories:
    if c in mapping:
        print(c, "->", mapping[c])

Burgers -> {'Count': 5636, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Burgers', 'Keep': 'Yes', 'Notes': nan}
Fast Food -> {'Count': 6472, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Fast Food', 'Keep': 'Yes', 'Notes': nan}
Sandwiches -> {'Count': 8366, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Sandwiches', 'Keep': 'Yes', 'Notes': nan}
Ice Cream & Frozen Yogurt -> {'Count': 1088, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Ice Cream & Frozen Yogurt', 'Keep': 'Yes', 'Notes': nan}


In [10]:
def extract_features(category_string, mapping):

    features = {
        "Cuisine": set(),
        "Restaurant Type": set(),
        "Experience": set()
    }

    categories = [
        category.strip()
        for category in category_string.split(",")
    ]

    for category in categories:

        if category in mapping:

            info = mapping[category]

            feature_type = info["Feature Type"]

            value = info["Standardized Value"]

            if feature_type in features and pd.notna(value):
                features[feature_type].add(value)

    return {
        key: list(value)
        for key, value in features.items()
    }

In [11]:
restaurant = restaurants.iloc[0]["categories"]

extract_features(restaurant,mapping)



{'Cuisine': [],
 'Restaurant Type': ['Burgers',
  'Ice Cream & Frozen Yogurt',
  'Fast Food',
  'Sandwiches'],
 'Experience': []}

In [12]:
restaurants["Extracted Features"] = restaurants["categories"].apply(
    lambda x: extract_features(x, mapping)
)

In [13]:
restaurants[["name", "Extracted Features"]].head()

,name,Extracted Features
0,Sonic Drive-In,"{'Cuisine': [], 'Restaurant Type': ['Burgers',..."
1,Sonic Drive-In,"{'Cuisine': [], 'Restaurant Type': ['Burgers',..."
2,Super Dog,"{'Cuisine': [], 'Restaurant Type': ['Hot Dogs'..."
3,The Green Pheasant,"{'Cuisine': ['Japanese'], 'Restaurant Type': [..."
4,Domino's Pizza,"{'Cuisine': [], 'Restaurant Type': ['Sandwiche..."


In [14]:
restaurants["Restaurant Type"] = restaurants["Extracted Features"].apply(
    lambda features: features["Restaurant Type"]
)

restaurants["Experience"] = restaurants["Extracted Features"].apply(
    lambda features: features["Experience"]
)

restaurants["Cuisine"] = restaurants["Extracted Features"].apply(
    lambda features: features["Cuisine"]
)

In [15]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

cuisine_mlb = mlb.fit_transform(restaurants["Cuisine"])

cuisine_mlb

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4352, 69))

In [16]:
Cuisine = mlb.inverse_transform(cuisine_mlb)

Cuisine

[(),
 (),
 (),
 ('Japanese',),
 (),
 ('American',),
 (),
 (),
 ('Mexican',),
 (),
 (),
 ('American',),
 ('Italian',),
 ('Latin American', 'Mexican'),
 ('Caribbean',),
 (),
 (),
 ('Italian',),
 (),
 ('Indian',),
 ('American',),
 ('American',),
 ('Indian',),
 ('Mediterranean',),
 (),
 (),
 (),
 ('Italian',),
 ('American',),
 (),
 ('Italian',),
 ('American',),
 ('American',),
 (),
 ('American',),
 ('Latin American',),
 ('Asian Fusion',),
 ('Chinese',),
 ('American',),
 ('American',),
 (),
 (),
 ('Greek', 'Mediterranean'),
 ('American',),
 ('Mexican',),
 ('American',),
 ('Mexican',),
 ('American', 'Cajun/Creole'),
 ('Chinese', 'Japanese', 'Korean'),
 ('American',),
 ('Chinese',),
 ('American',),
 ('Laotian', 'Thai', 'Vietnamese'),
 (),
 ('American',),
 ('Asian Fusion', 'Chinese'),
 ('Japanese',),
 ('Mexican',),
 (),
 (),
 (),
 (),
 ('American', 'Greek', 'Mediterranean'),
 ('American',),
 ('Mexican', 'New Mexican'),
 ('Asian Fusion', 'Chinese', 'Laotian', 'Oriental', 'Thai'),
 ('Asian Fusio

In [17]:
cuisine_df = pd.DataFrame(cuisine_mlb,columns=mlb.classes_)

In [18]:
restaurants["Restaurant Type"].apply(type).value_counts()

Restaurant Type
<class 'list'>    4352
Name: count, dtype: int64

In [19]:
all_types = set()

for lst in restaurants["Restaurant Type"]:
    all_types.update(type(x) for x in lst)

all_types

{str}

In [20]:
restaurants[
    restaurants["Restaurant Type"].apply(
        lambda lst: any(not isinstance(x, str) for x in lst)
    )
][["name", "Restaurant Type"]]

,name,Restaurant Type


In [21]:
restaurant_type_mlb = mlb.fit_transform(restaurants["Restaurant Type"])

restaurant_type = mlb.inverse_transform(restaurant_type_mlb)

restaurant_type_df = pd.DataFrame(restaurant_type_mlb, columns=mlb.classes_)

restaurant_type_df

,Acai Bowls,Alcoholic Beverages,Bagels,Bakery,Barbeque,Beer,Breakfast & Brunch,Bubble Tea,Buffets,Burgers,...,Tacos,Tapas/Small Plates,Tea,Teppanyaki,Vegan,Vegetarian,Waffles,Whiskey,Wine & Spirits,Wraps
0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4347,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4348,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4349,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4350,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
experience_mlb = mlb.fit_transform(restaurants["Experience"])

experience = mlb.inverse_transform(experience_mlb)

experience_df = pd.DataFrame(experience_mlb, columns=mlb.classes_)

experience_df

,Beer,Lounges,Wine
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
...,...,...,...
4347,0,0,0
4348,0,0,0
4349,0,0,0
4350,0,0,0


In [23]:
numeric_features_df = restaurants[
    [
        "latitude",
        "longitude",
        "stars",
        "review_count"
    ]
]

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(numeric_features_df[["stars","review_count"]])

scaled_numeric

scaled_numeric_df = pd.DataFrame(scaled_numeric, columns = ['stars', 'review_count'])

scaled_numeric_df


,stars,review_count
0,-1.650558,-0.410940
1,-2.209999,-0.392949
2,0.587208,-0.410940
3,0.587208,0.286187
4,0.027766,-0.401944
...,...,...
4347,0.027766,0.236714
4348,0.587208,-0.410940
4349,1.146649,-0.136586
4350,-0.531675,-0.410940


In [25]:
CUISINE_WEIGHT = 3.0
TYPE_WEIGHT = 2.0
EXPERIENCE_WEIGHT = 1.5
NUMERIC_WEIGHT = 1.0

In [26]:
weighted_cuisine = cuisine_df * CUISINE_WEIGHT

weighted_type = restaurant_type_df * TYPE_WEIGHT

weighted_experience = experience_df * EXPERIENCE_WEIGHT

weighted_numeric = scaled_numeric_df * NUMERIC_WEIGHT

In [27]:
recommendation_feature_matrix = pd.concat(
    [
        weighted_cuisine,
        weighted_type,
        weighted_experience,
        weighted_numeric
    ],
    axis=1
)

In [28]:
def find_restaurant(
    restaurant_name,
    city,
    restaurants
):

    selected_index = restaurants[
        (restaurants["name"] == restaurant_name) &
        (restaurants["city"] == city)
    ].index[0]

    return selected_index

In [29]:
def get_feature_vector(
    selected_index,
    recommendation_feature_matrix
):

    selected_vector = recommendation_feature_matrix.loc[
        [selected_index]
    ]

    return selected_vector

In [30]:

def calculate_similarity(
    selected_vector,
    recommendation_feature_matrix
):

    similarity_scores = cosine_similarity(
        selected_vector,
        recommendation_feature_matrix
    )

    similarity_df = pd.DataFrame(
        similarity_scores.T,
        columns=["similarity"]
    )

    return similarity_df

In [31]:
def get_top_recommendations(
    similarity_df,
    restaurants,
    selected_index,
    top_n=10,
    exclude_same_chain=False
):

    filtered_df = similarity_df.drop(selected_index)

    if exclude_same_chain:
        selected_name = restaurants.loc[selected_index, "name"]

        filtered_df = filtered_df[
            restaurants.loc[filtered_df.index, "name"] != selected_name
        ]

    top_similarity_df = filtered_df.nlargest(
        top_n,
        "similarity"
    )

    return top_similarity_df

In [32]:
def format_recommendations(
    top_similarity_df,
    restaurants
):

    recommendation_table = restaurants.loc[
        top_similarity_df.index
    ]

    recommendation_table = recommendation_table[
        [
            "name",
            "city",
            "state",
            "stars",
            "review_count"
        ]
    ]

    recommendation_table = pd.concat(
        [recommendation_table, top_similarity_df],
        axis=1
    )

    return recommendation_table

In [33]:
def recommend_restaurants(
    restaurant_name,
    city,
    restaurants,
    recommendation_feature_matrix,
    top_n=10
):

    selected_index = find_restaurant(
        restaurant_name,
        city,
        restaurants
    )

    selected_vector = get_feature_vector(
        selected_index,
        recommendation_feature_matrix
    )

    similarity_df = calculate_similarity(
        selected_vector,
        recommendation_feature_matrix
    )

    top_similarity_df = get_top_recommendations(
    similarity_df=similarity_df,
    restaurants=restaurants,
    selected_index=selected_index,
    top_n=top_n,
    exclude_same_chain=True
    )   

    recommendation_table = format_recommendations(
        top_similarity_df,
        restaurants
    )

    return recommendation_table

In [34]:
recommend_restaurants(
    restaurant_name="Sonic Drive-In",
    city="Ashland City",
    restaurants=restaurants,
    recommendation_feature_matrix=recommendation_feature_matrix,
    top_n=10
)

,name,city,state,stars,review_count,similarity
1963,Dairy Queen Grill & Chill,Bellevue,TN,2.0,28,0.887563
2561,Arby's,Smyrna,TN,1.5,10,0.881133
1931,Wendy's,Franklin,TN,1.5,23,0.880894
734,Arby's,Nashville,TN,2.5,12,0.879323
658,Wendy's,Goodlettsville,TN,2.5,13,0.879320
3191,Dairy Queen Grill & Chill,Bellevue,TN,2.5,15,0.879309
2709,Dairy Queen Grill & Chill,Nashville,TN,2.5,17,0.879294
2486,Dairy Queen Grill & Chill,Joelton,TN,3.0,6,0.850603
1212,Dairy Queen Grill & Chill,Goodlettsville,TN,3.0,5,0.850597
4103,Arby's,Nashville,TN,3.0,16,0.850593
